# Проверка симуляции mouse NovaSeq после фильтрации аннотации

Подробная проверка по набору истины для ветки mouse fastp:

```text
results/ERP003950/
└── simulated/
    └── insilicoseq_150bp_novaseq_post_annotation_filtered/
```

Ноутбук дополняет следующие проверки:

- раздел **14. Итоговая проверка** в симуляционном ноутбуке;
- общий этап FastQC/MultiQC в `qc.ipynb`.

Все шесть samples мыши независимо проверяются относительно точных фрагментов,
использованных InSilicoSeq, и при необходимости относительно исходных V…J templates после фильтрации.

In [ ]:
import csv, gzip, json, os, re, shutil, subprocess
from collections import Counter
from pathlib import Path

DATASET = "ERP003950"
BRANCH = "insilicoseq_150bp_novaseq_post_annotation_filtered"

SAMPLES = [
    "ERR346596",
    "ERR346597",
    "ERR346598",
    "ERR346599",
    "ERR346600",
    "ERR346601",
]
EXPECTED_LOCUS = {sample: "IGH" for sample in SAMPLES}

# Сохраняем общий объём выравнивания сопоставимым с проверкой human
# (6 x 50k = 300k пар ридов). Для проверки всех пар установите None.
VALIDATION_PAIRS_PER_SAMPLE = 50_000

# Mapping на уровне template информативен, но по своей природе менее однозначен, чем
# mapping точных фрагментов, поскольку BCR-перестройки содержат гомологичные последовательности.
RUN_TEMPLATE_ALIGNMENT = True

NPROC = 8
FORCE = False

def resolve_root():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))

    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ]

    start = Path.cwd().resolve()
    candidates += [start, *start.parents]

    seen = set()
    for root in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root

    raise FileNotFoundError(
        f"Cannot locate results/{DATASET}/simulated/{BRANCH}. "
        "Set BCR_VOLUME if the project lives elsewhere."
    )

ROOT = resolve_root()
SIM_DIR = ROOT / "results" / DATASET / "simulated" / BRANCH

TRUTH_DIR = SIM_DIR / "00_primary_truth"
PCR1_DIR = SIM_DIR / "01_pcr1"
FRAG_DIR = SIM_DIR / "02_fragmentation"
PCR2_DIR = SIM_DIR / "03_pcr2"
ALLOC_DIR = SIM_DIR / "04_read_allocation"
FASTQ_DIR = SIM_DIR / "06_fastq_pe150"
SIM_QC_DIR = SIM_DIR / "qc"

VALIDATION_DIR = SIM_DIR / "validation"
SUBSET_DIR = VALIDATION_DIR / "subset_fastq"
FRAG_ALIGN_DIR = VALIDATION_DIR / "fragment_alignment"
TEMPLATE_ALIGN_DIR = VALIDATION_DIR / "template_alignment"

for d in (
    VALIDATION_DIR,
    SUBSET_DIR,
    FRAG_ALIGN_DIR,
    TEMPLATE_ALIGN_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

FINAL_QC = SIM_QC_DIR / "final_qc.tsv"
if not FINAL_QC.exists():
    raise FileNotFoundError(
        f"{FINAL_QC} is missing. Run the simulation through section 14 first."
    )

for tool in ("bowtie2", "bowtie2-build", "samtools"):
    if not shutil.which(tool):
        raise RuntimeError(f"{tool} not found in PATH")

print("ROOT:", ROOT)
print("DATASET:", DATASET)
print("BRANCH:", BRANCH)
print("SIM_DIR:", SIM_DIR)
print("samples:", ", ".join(SAMPLES))

## 1. Внутренние инварианты симуляции

Для каждого sample независимо проверяется:

- число итоговых пар FASTQ равно бюджету симуляции;
- число R1 и R2 совпадает;
- каждый итоговый рид имеет длину ровно 150 нт;
- фрагментация сохраняет число молекул PCR1;
- сумма распределённых ридов точно равна ожидаемому числу пар;
- каждый распределённый фрагмент относится к ожидаемому локусу мыши (`IGH`).

In [ ]:
def read_tsv(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def count_fastq(path):
    n = 0
    lengths = set()

    with gzip.open(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                break
            seq = h.readline().rstrip("\r\n")
            plus = h.readline()
            qual = h.readline().rstrip("\r\n")

            if not qual or not head.startswith("@") or not plus.startswith("+"):
                raise RuntimeError(f"Malformed FASTQ: {path}")
            if len(seq) != len(qual):
                raise RuntimeError(
                    f"Sequence/quality length mismatch in {path}: {head.strip()}"
                )

            n += 1
            lengths.add(len(seq))

    return n, sorted(lengths)

final_rows = read_tsv(FINAL_QC)
final_by_sample = {row["sample"]: row for row in final_rows}

missing = sorted(set(SAMPLES) - set(final_by_sample))
if missing:
    raise RuntimeError(f"Samples missing from final_qc.tsv: {missing}")

invariant_rows = []

for sample in SAMPLES:
    q = final_by_sample[sample]
    if str(q["valid"]).strip().lower() != "true":
        raise RuntimeError(f"{sample}: simulation final_qc.tsv is not valid")

    expected_pairs = int(q["expected_pairs"])

    r1 = FASTQ_DIR / f"{sample}_R1.fastq.gz"
    r2 = FASTQ_DIR / f"{sample}_R2.fastq.gz"
    pcr1_tsv = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    fragment_tsv = FRAG_DIR / f"{sample}_fragments.tsv"
    allocation_tsv = ALLOC_DIR / f"{sample}_allocation.tsv"
    fragment_fasta = ALLOC_DIR / f"{sample}_selected_fragments.fasta"
    template_fasta = TRUTH_DIR / f"{sample}_templates.fasta"

    for p in (
        r1,
        r2,
        pcr1_tsv,
        fragment_tsv,
        allocation_tsv,
        fragment_fasta,
        template_fasta,
    ):
        if not p.exists():
            raise FileNotFoundError(p)

    n1, l1 = count_fastq(r1)
    n2, l2 = count_fastq(r2)

    if n1 != n2 or n1 != expected_pairs:
        raise RuntimeError(
            f"{sample}: pair-count mismatch: "
            f"expected={expected_pairs:,}, R1={n1:,}, R2={n2:,}"
        )
    if l1 != [150] or l2 != [150]:
        raise RuntimeError(
            f"{sample}: expected PE150, observed R1={l1}, R2={l2}"
        )

    pcr1 = read_tsv(pcr1_tsv)
    fragments = read_tsv(fragment_tsv)
    allocation = read_tsv(allocation_tsv)

    pcr1_molecules = sum(int(r["pcr_copies"]) for r in pcr1)
    fragment_input_molecules = sum(
        int(r["fragment_input_copies"]) for r in fragments
    )

    if pcr1_molecules != fragment_input_molecules:
        raise RuntimeError(
            f"{sample}: fragmentation mass is not conserved: "
            f"PCR1={pcr1_molecules:,}, "
            f"fragment_input={fragment_input_molecules:,}"
        )

    allocated_pairs = sum(int(r["simulated_read_pairs"]) for r in allocation)
    if allocated_pairs != expected_pairs:
        raise RuntimeError(
            f"{sample}: allocation={allocated_pairs:,} "
            f"!= expected={expected_pairs:,}"
        )

    loci = sorted({
        str(r.get("locus", "")).strip()
        for r in allocation
        if int(r["simulated_read_pairs"]) > 0
    })
    if loci != [EXPECTED_LOCUS[sample]]:
        raise RuntimeError(
            f"{sample}: unexpected allocated loci {loci}; "
            f"expected {[EXPECTED_LOCUS[sample]]}"
        )

    invariant_rows.append({
        "sample": sample,
        "expected_locus": EXPECTED_LOCUS[sample],
        "expected_pairs": expected_pairs,
        "R1_reads": n1,
        "R2_reads": n2,
        "R1_lengths": ",".join(map(str, l1)),
        "R2_lengths": ",".join(map(str, l2)),
        "pcr1_molecules": pcr1_molecules,
        "fragment_input_molecules": fragment_input_molecules,
        "fragmentation_molecule_conserved": True,
        "allocated_pairs": allocated_pairs,
        "allocated_loci": ",".join(loci),
        "valid": True,
    })

invariants_path = VALIDATION_DIR / "sample_invariants.tsv"
with open(invariants_path, "w", newline="") as h:
    w = csv.DictWriter(
        h,
        fieldnames=list(invariant_rows[0]),
        delimiter="\t",
    )
    w.writeheader()
    w.writerows(invariant_rows)

for row in invariant_rows:
    print(row)
print("wrote", invariants_path)

## 2. Детерминированные парные подвыборки для проверки

Чтобы не выравнивать все риды при стандартной проверке, ноутбук берёт равномерную
детерминированную подвыборку из каждого sample.

По умолчанию:

```python
VALIDATION_PAIRS_PER_SAMPLE = 50_000
```

Это соответствует примерно 300,000 парам PE суммарно для шести samples мыши.

Для полной проверки установите `VALIDATION_PAIRS_PER_SAMPLE = None`.

In [ ]:
def paired_subset(r1, r2, out1, out2, total_pairs, target_pairs):
    if target_pairs is None or target_pairs >= total_pairs:
        return r1, r2, total_pairs

    stride = max(1, total_pairs // target_pairs)
    selected = 0

    tmp1 = Path(str(out1) + ".tmp")
    tmp2 = Path(str(out2) + ".tmp")

    def read_record(h):
        a = h.readline()
        if not a:
            return None
        return (a, h.readline(), h.readline(), h.readline())

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2, \
         gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:

        i = 0
        while True:
            a = read_record(h1)
            b = read_record(h2)

            if a is None or b is None:
                if a is not None or b is not None:
                    raise RuntimeError(
                        f"Mate FASTQs ended at different positions: {r1}, {r2}"
                    )
                break

            if i % stride == 0 and selected < target_pairs:
                o1.writelines(a)
                o2.writelines(b)
                selected += 1

            i += 1

    tmp1.replace(out1)
    tmp2.replace(out2)
    return out1, out2, selected

validation_inputs = {}

for row in invariant_rows:
    sample = row["sample"]
    total_pairs = int(row["expected_pairs"])

    r1 = FASTQ_DIR / f"{sample}_R1.fastq.gz"
    r2 = FASTQ_DIR / f"{sample}_R2.fastq.gz"

    sub1 = SUBSET_DIR / f"{sample}_R1.validation.fastq.gz"
    sub2 = SUBSET_DIR / f"{sample}_R2.validation.fastq.gz"

    if FORCE or not (sub1.exists() and sub2.exists()):
        vr1, vr2, n = paired_subset(
            r1,
            r2,
            sub1,
            sub2,
            total_pairs,
            VALIDATION_PAIRS_PER_SAMPLE,
        )
    else:
        vr1, vr2 = sub1, sub2
        n1, _ = count_fastq(sub1)
        n2, _ = count_fastq(sub2)
        if n1 != n2:
            raise RuntimeError(f"{sample}: cached validation mates differ")
        n = n1

    validation_inputs[sample] = {
        "R1": vr1,
        "R2": vr2,
        "pairs": n,
    }

    print(sample, "validation pairs:", f"{n:,}")

## 3. Выравнивание ридов на точные выбранные фрагменты

Это наиболее строгая проверка симуляции секвенирования: InSilicoSeq сгенерировал
риды именно из этих последовательностей фрагментов.

Для каждого sample ноутбук сообщает:

- долю mapped;
- долю properly paired;
- частоту ошибок из `samtools stats`.

In [ ]:
def run(cmd, log=None):
    cmd = list(map(str, cmd))
    print("[run]", " ".join(cmd), flush=True)

    if log is None:
        subprocess.run(cmd, check=True)
    else:
        with open(log, "w") as h:
            subprocess.run(
                cmd,
                stdout=h,
                stderr=subprocess.STDOUT,
                check=True,
            )

def ensure_index(reference, prefix):
    marker = Path(str(prefix) + ".1.bt2")
    marker_large = Path(str(prefix) + ".1.bt2l")

    if FORCE or not (marker.exists() or marker_large.exists()):
        for p in prefix.parent.glob(prefix.name + "*.bt2*"):
            p.unlink()

        run(
            [
                "bowtie2-build",
                "--threads",
                str(NPROC),
                reference,
                prefix,
            ],
            prefix.parent / "bowtie2_build.log",
        )

    return prefix

def align_pair(sample, reference, outdir, label):
    outdir.mkdir(parents=True, exist_ok=True)

    index = ensure_index(reference, outdir / "index")
    sam = outdir / f"{label}.sam"
    bam = outdir / f"{label}.bam"

    vr1 = validation_inputs[sample]["R1"]
    vr2 = validation_inputs[sample]["R2"]

    if FORCE or not bam.exists():
        run(
            [
                "bowtie2",
                "--very-sensitive-local",
                "-p",
                str(NPROC),
                "-x",
                index,
                "-1",
                vr1,
                "-2",
                vr2,
                "-S",
                sam,
            ],
            outdir / "bowtie2_align.log",
        )

        run([
            "samtools",
            "sort",
            "-@",
            str(NPROC),
            "-o",
            bam,
            sam,
        ])
        run(["samtools", "index", bam])
        sam.unlink(missing_ok=True)

    return bam

def flagstat_metrics(bam):
    txt = subprocess.run(
        ["samtools", "flagstat", str(bam)],
        capture_output=True,
        text=True,
        check=True,
    ).stdout

    mapped_pct = None
    properly_paired_pct = None

    for line in txt.splitlines():
        if " mapped (" in line and "primary mapped" not in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                mapped_pct = float(m.group(1))

        if " properly paired (" in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                properly_paired_pct = float(m.group(1))

    return mapped_pct, properly_paired_pct, txt

def samtools_error_rate(bam):
    txt = subprocess.run(
        ["samtools", "stats", str(bam)],
        capture_output=True,
        text=True,
        check=True,
    ).stdout

    for line in txt.splitlines():
        if "error rate:" in line:
            parts = line.split("\t")
            for i, x in enumerate(parts):
                if x.strip() == "error rate:" and i + 1 < len(parts):
                    return float(parts[i + 1])

    return None

fragment_bams = {}
fragment_alignment_metrics = {}

for sample in SAMPLES:
    ref = ALLOC_DIR / f"{sample}_selected_fragments.fasta"
    outdir = FRAG_ALIGN_DIR / sample

    bam = align_pair(
        sample,
        ref,
        outdir,
        "selected_fragments",
    )
    fragment_bams[sample] = bam

    mapped, proper, _ = flagstat_metrics(bam)
    error_rate = samtools_error_rate(bam)

    fragment_alignment_metrics[sample] = {
        "mapped_pct": mapped,
        "properly_paired_pct": proper,
        "error_rate": error_rate,
    }

    print(
        sample,
        "mapped%=", mapped,
        "properly_paired%=", proper,
        "error_rate=", error_rate,
    )

## 4. Соответствие исходному фрагменту

Одного mapping недостаточно для BCR-репертуара, поскольку гомологичные последовательности
могут выравниваться на несколько референсов.

Имя рида InSilicoSeq содержит идентификатор исходного фрагмента, поэтому на этом
этапе ожидаемый фрагмент из имени рида сравнивается с основным назначением Bowtie2.

Рассчитываются две доли:

- **exact fragment origin rate** — восстановлен точный ID фрагмента;
- **sequence-equivalent fragment origin rate** — допускается другой ID фрагмента,
  только если его нуклеотидная последовательность полностью совпадает с истинным фрагментом.

In [ ]:
import pysam

def iter_fasta(path):
    with open(path) as h:
        name = None
        chunks = []

        for line in h:
            line = line.rstrip("\r\n")

            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)

        if name is not None:
            yield name, "".join(chunks)

# ID фрагментов текущей симуляции заканчиваются на "_frag<integer>".
QNAME_RE = re.compile(r"^(.*_frag\d+)_\d+_\d+(?:/[12])?$")

def expected_fragment_id(qname):
    m = QNAME_RE.match(qname)
    return m.group(1) if m else None

def fragment_origin_metrics(
    bam,
    fragment_sequences,
    max_primary_reads=500_000,
):
    parsed = 0
    exact = 0
    equivalent = 0
    primary = 0

    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue

            primary += 1
            expected = expected_fragment_id(r.query_name)

            if expected is not None and expected in fragment_sequences:
                parsed += 1

                if expected == r.reference_name:
                    exact += 1
                    equivalent += 1
                elif (
                    r.reference_name in fragment_sequences
                    and fragment_sequences[expected]
                    == fragment_sequences[r.reference_name]
                ):
                    equivalent += 1

            if primary >= max_primary_reads:
                break

    return {
        "primary_reads_checked": primary,
        "qname_origin_parse_rate": (
            parsed / primary if primary else None
        ),
        "exact_fragment_origin_rate": (
            exact / parsed if parsed else None
        ),
        "sequence_equivalent_fragment_origin_rate": (
            equivalent / parsed if parsed else None
        ),
    }

fragment_origin = {}

for sample in SAMPLES:
    ref = ALLOC_DIR / f"{sample}_selected_fragments.fasta"
    fragment_sequences = dict(iter_fasta(ref))

    metrics = fragment_origin_metrics(
        fragment_bams[sample],
        fragment_sequences,
    )
    fragment_origin[sample] = metrics

    print(sample)
    print(json.dumps(metrics, indent=2))

## 5. Необязательное выравнивание на исходный набор истины V…J

Исходные templates — это точные V…J-последовательности после фильтрации, до PCR
и фрагментации.

Проверка подтверждает связь синтетических ридов с исходным набором истины репертуара,
но mapping здесь может быть менее однозначным, чем для точных выбранных фрагментов.

In [ ]:
template_alignment_metrics = {}

if RUN_TEMPLATE_ALIGNMENT:
    for sample in SAMPLES:
        ref = TRUTH_DIR / f"{sample}_templates.fasta"
        outdir = TEMPLATE_ALIGN_DIR / sample

        bam = align_pair(
            sample,
            ref,
            outdir,
            "primary_templates",
        )

        mapped, proper, _ = flagstat_metrics(bam)
        error_rate = samtools_error_rate(bam)

        template_alignment_metrics[sample] = {
            "mapped_pct": mapped,
            "properly_paired_pct": proper,
            "error_rate": error_rate,
        }

        print(
            sample,
            "template mapped%=", mapped,
            "properly_paired%=", proper,
            "error_rate=", error_rate,
        )
else:
    template_alignment_metrics = {
        sample: None
        for sample in SAMPLES
    }
    print("Template alignment skipped")

## 6. Итоговые отчёты проверки

Выходы:

```text
validation/
├── sample_invariants.tsv
├── alignment_metrics.tsv
├── validation_summary.json
├── subset_fastq/
├── fragment_alignment/
│   ├── ERR346596/
│   ├── ...
│   └── ERR346601/
└── template_alignment/
    ├── ERR346596/
    ├── ...
    └── ERR346601/
```

`validation_summary.json` — основной машиночитаемый отчёт.

In [ ]:
alignment_rows = []
sample_summaries = {}

for row in invariant_rows:
    sample = row["sample"]
    frag = {
        **fragment_alignment_metrics[sample],
        **fragment_origin[sample],
    }
    tmpl = template_alignment_metrics[sample]

    alignment_row = {
        "sample": sample,
        "expected_locus": EXPECTED_LOCUS[sample],
        "validation_pairs": validation_inputs[sample]["pairs"],
        "fragment_mapped_pct": frag["mapped_pct"],
        "fragment_properly_paired_pct": frag["properly_paired_pct"],
        "fragment_error_rate": frag["error_rate"],
        "qname_origin_parse_rate": frag["qname_origin_parse_rate"],
        "exact_fragment_origin_rate": frag["exact_fragment_origin_rate"],
        "sequence_equivalent_fragment_origin_rate": (
            frag["sequence_equivalent_fragment_origin_rate"]
        ),
        "template_mapped_pct": (
            tmpl["mapped_pct"] if tmpl is not None else None
        ),
        "template_properly_paired_pct": (
            tmpl["properly_paired_pct"] if tmpl is not None else None
        ),
        "template_error_rate": (
            tmpl["error_rate"] if tmpl is not None else None
        ),
    }
    alignment_rows.append(alignment_row)

    sample_summaries[sample] = {
        "expected_locus": EXPECTED_LOCUS[sample],
        "expected_pairs": int(row["expected_pairs"]),
        "validation_pairs": validation_inputs[sample]["pairs"],
        "internal_invariants": {
            "R1_reads": int(row["R1_reads"]),
            "R2_reads": int(row["R2_reads"]),
            "read_length": 150,
            "pcr1_molecules": int(row["pcr1_molecules"]),
            "fragment_input_molecules": int(
                row["fragment_input_molecules"]
            ),
            "fragmentation_molecule_conserved": True,
            "allocated_pairs": int(row["allocated_pairs"]),
            "valid": True,
        },
        "fragment_alignment": frag,
        "template_alignment": tmpl,
    }

alignment_path = VALIDATION_DIR / "alignment_metrics.tsv"
with open(alignment_path, "w", newline="") as h:
    w = csv.DictWriter(
        h,
        fieldnames=list(alignment_rows[0]),
        delimiter="\t",
    )
    w.writeheader()
    w.writerows(alignment_rows)

def weighted_mean(metric_key):
    num = 0.0
    den = 0

    for row in alignment_rows:
        value = row[metric_key]
        if value is None:
            continue
        weight = int(row["validation_pairs"])
        num += float(value) * weight
        den += weight

    return num / den if den else None

summary = {
    "dataset": DATASET,
    "branch": BRANCH,
    "expected_locus": "IGH",
    "samples": SAMPLES,
    "validation_pairs_per_sample_setting": VALIDATION_PAIRS_PER_SAMPLE,
    "total_validation_pairs": sum(
        validation_inputs[s]["pairs"]
        for s in SAMPLES
    ),
    "all_internal_invariants_valid": all(
        bool(r["valid"])
        for r in invariant_rows
    ),
    "aggregate_weighted": {
        "fragment_mapped_pct": weighted_mean(
            "fragment_mapped_pct"
        ),
        "fragment_properly_paired_pct": weighted_mean(
            "fragment_properly_paired_pct"
        ),
        "fragment_error_rate": weighted_mean(
            "fragment_error_rate"
        ),
        "qname_origin_parse_rate": weighted_mean(
            "qname_origin_parse_rate"
        ),
        "exact_fragment_origin_rate": weighted_mean(
            "exact_fragment_origin_rate"
        ),
        "sequence_equivalent_fragment_origin_rate": weighted_mean(
            "sequence_equivalent_fragment_origin_rate"
        ),
        "template_mapped_pct": weighted_mean(
            "template_mapped_pct"
        ),
        "template_properly_paired_pct": weighted_mean(
            "template_properly_paired_pct"
        ),
        "template_error_rate": weighted_mean(
            "template_error_rate"
        ),
    },
    "per_sample": sample_summaries,
}

summary_path = VALIDATION_DIR / "validation_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2) + "\n"
)

print("wrote", invariants_path)
print("wrote", alignment_path)
print("wrote", summary_path)
print(json.dumps(summary["aggregate_weighted"], indent=2))

## Интерпретация

Структурная проверка считается успешной, если:

1. каждая строка в `sample_invariants.tsv` имеет `valid=True`;
2. для всех шести samples установлено `fragmentation_molecule_conserved=True`;
3. число распределённых пар ридов точно равно ожидаемому бюджету;
4. в распределении симуляции мыши присутствует только `IGH`;
5. доля mapping на уровне фрагментов очень высока;
6. доля sequence-equivalent origin очень высока.

`exact_fragment_origin_rate` может быть ниже sequence-equivalent rate, если
разные ID фрагментов имеют одинаковые нуклеотидные последовательности. Mapping на уровне
templates также может быть менее специфичным из-за гомологичных V/J-последовательностей.

После успешного выполнения этого ноутбука и общего `qc.ipynb` сгенерированные данные
PE150 готовы к тестам реконструкции репертуара, например TRUST4.